# Enhancing Financial Market Predictions with Financial News Sentiment

## Project topic

This project investigates whether financial news sentiment is related to short-term stock market returns for selected technology companies.

The analysis combines two independent data sources:

1. **Financial news headlines** from `analyst_ratings_processed.csv`
2. **Historical stock market prices** from `tech_stock_prices_2020_to_today.csv`


---

## Research question

**Can daily financial news sentiment help explain or predict daily stock returns?**

We will focus on three companies that are present in both datasets and have sufficient news coverage:

- **AAPL** — Apple
- **TSLA** — Tesla
- **NVDA** — Nvidia


## Mathematical background

### Daily return

Daily return measures the relative change in closing price between two consecutive trading days:

$$
R_t = \frac{P_t - P_{t-1}}{P_{t-1}}
$$

where:

- $P_t$ is the closing price on day $t$
- $P_{t-1}$ is the closing price on the previous trading day

### Sentiment score

Each headline is converted into a numerical sentiment score:

$$
S_i \in [-1, 1]
$$

where:

- negative values indicate negative sentiment
- values close to zero indicate neutral sentiment
- positive values indicate positive sentiment

Daily sentiment is calculated as the average sentiment of all headlines for a company on a given day:

$$
\bar{S}_{t} = \frac{1}{n}\sum_{i=1}^{n} S_i
$$

### Correlation

The Pearson correlation coefficient measures the linear relationship between sentiment and returns:

$$
\rho_{S,R} = \frac{Cov(S, R)}{\sigma_S \sigma_R}
$$

### Regression

A simple regression model can be written as:

$$
R_t = \beta_0 + \beta_1 S_t + \epsilon_t
$$

where:

- $R_t$ is the stock return
- $S_t$ is the sentiment score
- $\beta_0$ is the intercept
- $\beta_1$ measures the relationship between sentiment and returns
- $\epsilon_t$ is the error term

In [31]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import re
import math
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

## 1. Data loading

The notebook expects the following files to be available in the same folder as the notebook:

- `analyst_ratings_processed.csv`
- `tech_stock_prices_2020_to_today.csv`

If the notebook is executed in the original working environment, it will also try `/mnt/data`.

In [32]:
DATA_DIR = Path(".")
# if not (DATA_DIR / "analyst_ratings_processed.csv").exists():
#     DATA_DIR = Path("/mnt/data")

NEWS_PATH = DATA_DIR / "analyst_ratings_processed.csv"
STOCK_PATH = DATA_DIR / "tech_stock_prices_2020_to_today.csv"

print("News dataset path:", NEWS_PATH)
print("Stock dataset path:", STOCK_PATH)

News dataset path: analyst_ratings_processed.csv
Stock dataset path: tech_stock_prices_2020_to_today.csv


In [33]:
news_raw = pd.read_csv(
    NEWS_PATH,
    usecols=["title", "date", "stock"]
)

stock_raw = pd.read_csv(STOCK_PATH)

print("News shape:", news_raw.shape)
print("Stock shape:", stock_raw.shape)

display(news_raw.head())
display(stock_raw.head())

News shape: (1400466, 3)
Stock shape: (23160, 18)


,title,date,stock
0,Stocks That Hit 52-Week Highs On Friday,2020-06-05 10:30:00-04:00,A
1,Stocks That Hit 52-Week Highs On Wednesday,2020-06-03 10:45:00-04:00,A
2,71 Biggest Movers From Friday,2020-05-26 04:30:00-04:00,A
3,46 Stocks Moving In Friday's Mid-Day Session,2020-05-22 12:45:00-04:00,A
4,B of A Securities Maintains Neutral on Agilent...,2020-05-22 11:38:00-04:00,A


,index,Date,Open,High,Low,Close,Adj Close,Volume,Ticker,Dividends,Stock Splits,P/E Ratio,Market Cap,Price/Sales Ratio,Price/Book Ratio,Dividend Yield,Daily Return,20-Day MA
0,0,2020-01-02,74.059998,75.150002,73.797501,75.087502,72.960464,135480400,AAPL,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2020-01-03,74.287498,75.144997,74.125000,74.357498,72.251122,146322800,AAPL,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,2020-01-06,73.447502,74.989998,73.187500,74.949997,72.826859,118387200,AAPL,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,2020-01-07,74.959999,75.224998,74.370003,74.597504,72.484352,108872000,AAPL,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,2020-01-08,74.290001,76.110001,74.290001,75.797501,73.650352,132079200,AAPL,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Initial data inspection

In [34]:
print("News columns:")
print(news_raw.columns)

print("\nStock columns:")
print(stock_raw.columns)

print("\nAvailable stock tickers in stock dataset:")
print(stock_raw["Ticker"].unique())

print("\nNews records for stock tickers available in the stock dataset:")
available_stock_tickers = stock_raw["Ticker"].unique()
display(news_raw[news_raw["stock"].isin(available_stock_tickers)]["stock"].value_counts())

News columns:
Index(['title', 'date', 'stock'], dtype='object')

Stock columns:
Index(['index', 'Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume',
       'Ticker', 'Dividends', 'Stock Splits', 'P/E Ratio', 'Market Cap',
       'Price/Sales Ratio', 'Price/Book Ratio', 'Dividend Yield',
       'Daily Return', '20-Day MA'],
      dtype='object')

Available stock tickers in stock dataset:
['AAPL' 'MSFT' 'AMZN' 'GOOGL' 'TSLA' 'META' 'NVDA' 'IBM' 'ORCL' 'INTC']

News records for stock tickers available in the stock dataset:


stock
NVDA     3133
ORCL     2695
TSLA     1930
GOOGL    1585
IBM      1083
AAPL      469
AMZN      330
INTC       10
Name: count, dtype: int64

## 3. Company selection

We will choose 3 companies. The selected companies are:

- AAPL
- TSLA
- NVDA

These companies are present in both datasets and have enough financial news records for some analysis.

In [35]:
selected_tickers = ["AAPL", "TSLA", "NVDA"]

news = news_raw[news_raw["stock"].isin(selected_tickers)].copy()
stock = stock_raw[stock_raw["Ticker"].isin(selected_tickers)].copy()

print("Filtered news shape:", news.shape)
print("Filtered stock shape:", stock.shape)

display(news["stock"].value_counts())
display(stock["Ticker"].value_counts())

Filtered news shape: (5532, 3)
Filtered stock shape: (6948, 18)


stock
NVDA    3133
TSLA    1930
AAPL     469
Name: count, dtype: int64

Ticker
AAPL    2316
TSLA    2316
NVDA    2316
Name: count, dtype: int64

## 4. Data cleaning

### News dataset cleaning

Required steps:

1. Remove missing titles, dates, and tickers.
2. Convert timestamps to UTC datetime.
3. Extract the calendar date for daily merging.
4. Clean text for sentiment analysis.

### Stock dataset cleaning

Required steps:

1. Convert dates to datetime.
2. Sort by company and date.
3. Ensure numerical columns have correct types.
4. Calculate daily returns if the existing column is incomplete.

In [36]:
news = news.dropna(subset=["title", "date", "stock"]).copy()

news["date"] = pd.to_datetime(news["date"], utc=True, errors="coerce")
news = news.dropna(subset=["date"]).copy()

news["Date"] = news["date"].dt.date
news["Date"] = pd.to_datetime(news["Date"])

news["title_clean"] = (
    news["title"]
    .astype(str)
    .str.lower()
    .str.replace(r"[^a-zA-Z\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

display(news.head())
print(news.info())

,title,date,stock,Date,title_clean
3666,Tech Stocks And FAANGS Strong Again To Start D...,2020-06-10 15:33:00+00:00,AAPL,2020-06-10,tech stocks and faangs strong again to start d...
3667,10 Biggest Price Target Changes For Wednesday,2020-06-10 12:14:00+00:00,AAPL,2020-06-10,biggest price target changes for wednesday
3668,"Benzinga Pro's Top 5 Stocks To Watch For Wed.,...",2020-06-10 11:53:00+00:00,AAPL,2020-06-10,benzinga pro s top stocks to watch for wed jun...
3669,"Deutsche Bank Maintains Buy on Apple, Raises P...",2020-06-10 11:19:00+00:00,AAPL,2020-06-10,deutsche bank maintains buy on apple raises pr...
3670,Apple To Let Users Trade In Their Mac Computer...,2020-06-10 10:27:00+00:00,AAPL,2020-06-10,apple to let users trade in their mac computer...


<class 'pandas.core.frame.DataFrame'>
Index: 5532 entries, 3666 to 1250208
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype              
---  ------       --------------  -----              
 0   title        5532 non-null   object             
 1   date         5532 non-null   datetime64[ns, UTC]
 2   stock        5532 non-null   object             
 3   Date         5532 non-null   datetime64[ns]     
 4   title_clean  5532 non-null   object             
dtypes: datetime64[ns, UTC](1), datetime64[ns](1), object(3)
memory usage: 259.3+ KB
None


In [37]:
stock["Date"] = pd.to_datetime(stock["Date"], errors="coerce")
stock = stock.dropna(subset=["Date", "Ticker", "Close"]).copy()

numeric_columns = [
    "Open", "High", "Low", "Close", "Adj Close", "Volume",
    "Daily Return", "20-Day MA"
]

for column in numeric_columns:
    if column in stock.columns:
        stock[column] = pd.to_numeric(stock[column], errors="coerce")

stock = stock.sort_values(["Ticker", "Date"])

stock["calculated_return"] = stock.groupby("Ticker")["Close"].pct_change()

if "Daily Return" in stock.columns:
    stock["Daily Return"] = stock["Daily Return"].fillna(stock["calculated_return"])
else:
    stock["Daily Return"] = stock["calculated_return"]

stock["return_next_day"] = stock.groupby("Ticker")["Daily Return"].shift(-1)
stock["close_lag_1"] = stock.groupby("Ticker")["Close"].shift(1)
stock["volume_change"] = stock.groupby("Ticker")["Volume"].pct_change()

display(stock.head())
display(stock[["Ticker", "Date", "Close", "Daily Return", "return_next_day"]].head(10))

,index,Date,Open,High,Low,Close,Adj Close,Volume,Ticker,Dividends,Stock Splits,P/E Ratio,Market Cap,Price/Sales Ratio,Price/Book Ratio,Dividend Yield,Daily Return,20-Day MA,calculated_return,return_next_day,close_lag_1,volume_change
0,0,2020-01-02,74.059998,75.150002,73.797501,75.087502,72.960464,135480400,AAPL,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN
11580,11580,2020-01-02,74.059998,75.150002,73.797501,75.087502,72.960464,135480400,AAPL,0.0,0.0,32.913242,3.287735e+12,8.526217,49.347332,0.0047,0.000000,NaN,0.000000,-0.009722,75.087502,0.000000
1,1,2020-01-03,74.287498,75.144997,74.125000,74.357498,72.251122,146322800,AAPL,0.0,0.0,NaN,NaN,NaN,NaN,NaN,-0.009722,NaN,-0.009722,-0.009722,75.087502,0.080029
11581,11581,2020-01-03,74.287498,75.144997,74.125000,74.357498,72.251122,146322800,AAPL,0.0,0.0,32.913242,3.287735e+12,8.526217,49.347332,0.0047,-0.009722,NaN,0.000000,0.007968,74.357498,0.000000
2,2,2020-01-06,73.447502,74.989998,73.187500,74.949997,72.826859,118387200,AAPL,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.007968,NaN,0.007968,0.007968,74.357498,-0.190918


,Ticker,Date,Close,Daily Return,return_next_day
0,AAPL,2020-01-02,75.087502,NaN,0.000000
11580,AAPL,2020-01-02,75.087502,0.000000,-0.009722
1,AAPL,2020-01-03,74.357498,-0.009722,-0.009722
11581,AAPL,2020-01-03,74.357498,-0.009722,0.007968
2,AAPL,2020-01-06,74.949997,0.007968,0.007968
11582,AAPL,2020-01-06,74.949997,0.007968,-0.004703
3,AAPL,2020-01-07,74.597504,-0.004703,-0.004703
11583,AAPL,2020-01-07,74.597504,-0.004703,0.016086
4,AAPL,2020-01-08,75.797501,0.016086,0.016086
11584,AAPL,2020-01-08,75.797501,0.016086,0.021241
